# LFW Grad-CAM — 01. Origin embedding and leave-one-out templates

Pass A에서 선택된 **모든 이미지**의 raw 512D, raw norm, unit embedding을
저장한 뒤 같은 split·identity 안에서 자기 자신을 제외한 template을
만듭니다. singleton과 identity 누락 표본도 행은 유지하며 명시적으로
부적격 처리합니다.


In [1]:
# cell 1 : 환경 설정 및 profile 검증
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(C:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_PROFILE = "arcface_ms1mv3_r100"     # arcface, adaface, magface 중 이번 실행 profile
MODE = "dev"               # 빠른 검증은 dev, 전체 논문 실행만 real
DATA_FRACTION = 1       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합과 random control의 재현 seed
EXECUTE_STAGE = True      # 필수 입력을 채우고 이 단계 계산 시에만 True
WRITE_OUTPUTS = True      # 새 immutable artifact 저장 시에만 True

all_profiles = CONFIG["models"]["selected_profiles"] + CONFIG["models"].get("bridge_profiles", [])
available_profiles = CONFIG["models"]["profiles"]
blocked_profiles = CONFIG["models"].get("blocked_profiles", [])

if MODEL_PROFILE in blocked_profiles:
    raise RuntimeError(f"차단된 profile입니다: {MODEL_PROFILE}")
if MODEL_PROFILE not in available_profiles:
    raise ValueError(f"지원하지 않는 MODEL_PROFILE: {MODEL_PROFILE}")
PROFILE = available_profiles[MODEL_PROFILE]
MODEL_FAMILY = PROFILE["family"]
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if MODE == "real" and DATA_FRACTION != 1.0:
    raise ValueError("real 모드는 DATA_FRACTION=1.0이어야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")


In [2]:
# cell 2 : 입력 경로 및 import
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    read_model_spec,
)
from research.explainability.gradcam import (
    prepare_population_saliency_inputs,
    write_prepared_population_artifact,
)
from research.runtime.hashing import sha256_file

# 명시적 경로 지정 (None일 경우 최신 run 경로 자동 탐색)
FREEZE_MANIFEST_PATH = None
SELECTED_MANIFEST_PATH = None
ALIGNED_FACES_NPY_PATH = None
PREPARED_ARTIFACT_OUTPUT_DIR = None

# CUDA 사용 가능 여부 자동 감지 (없으면 cpu 사용)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 자동 기본값 및 경로 탐색
if ALIGNED_FACES_NPY_PATH is None:
    ALIGNED_FACES_NPY_PATH = PROJECT_ROOT / CONFIG["datasets"]["lfw"]["aligned_crops"]["array_path"]

if FREEZE_MANIFEST_PATH is None or SELECTED_MANIFEST_PATH is None or PREPARED_ARTIFACT_OUTPUT_DIR is None:
    lfw_runs_root = PROJECT_ROOT / CONFIG["run"]["root"] / "lfw"
    latest_freeze_files = sorted(lfw_runs_root.rglob("freeze_manifest.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    if latest_freeze_files:
        latest_freeze = latest_freeze_files[0]
        run_dir = latest_freeze.parent
        if FREEZE_MANIFEST_PATH is None:
            FREEZE_MANIFEST_PATH = latest_freeze
        if SELECTED_MANIFEST_PATH is None:
            SELECTED_MANIFEST_PATH = run_dir / "selected_manifest.parquet"
        if PREPARED_ARTIFACT_OUTPUT_DIR is None:
            PREPARED_ARTIFACT_OUTPUT_DIR = run_dir / "prepared_population"


In [3]:
# cell 3 : Pass A embedding 추출 및 LOO template 생성
if EXECUTE_STAGE:
    paths = {
        "freeze": FREEZE_MANIFEST_PATH,
        "selected": SELECTED_MANIFEST_PATH,
        "aligned_faces": ALIGNED_FACES_NPY_PATH,
    }
    missing = [name for name, value in paths.items() if value is None]
    if missing:
        raise RuntimeError(f"입력 경로가 비어 있습니다: {missing}")
    freeze = json.loads(
        Path(paths["freeze"]).read_text(encoding="utf-8")
    )
    selected = pd.read_parquet(paths["selected"])

    # freeze manifest에 기록된 exact ModelSpec 사용
    model_entry = freeze["inputs"]["model_spec"]
    model_spec_path = Path(model_entry["path"])
    if sha256_file(model_spec_path) != model_entry["sha256"]:
        raise ValueError("동결된 ModelSpec hash가 변경됐습니다.")
    spec = read_model_spec(model_spec_path, verify_checkpoint=True)
    if spec.model_uid != freeze["model_uid"]:
        raise ValueError("freeze model_uid와 ModelSpec이 다릅니다.")
    if spec.checkpoint.sha256 != freeze["checkpoint_sha256"]:
        raise ValueError("checkpoint가 freeze 이후 변경됐습니다.")
    if spec.preprocessing.preprocess_hash != freeze["preprocess_hash"]:
        raise ValueError("preprocessing이 freeze 이후 변경됐습니다.")

    if len(selected) != int(freeze["selected_sample_count"]):
        raise ValueError("동결된 선택 표본 수와 manifest가 다릅니다.")

    source_faces = np.load(
        paths["aligned_faces"],
        mmap_mode="r",
        allow_pickle=False,
    )
    indices = selected["aligned_face_index"].to_numpy(dtype=np.int64)
    aligned_faces = np.asarray(source_faces[indices], dtype=np.uint8)
    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    prepared = prepare_population_saliency_inputs(
        adapter,
        aligned_faces,
        sample_ids=selected["sample_id"].astype(str),
        identity_ids=selected["identity_id"],
        scope_ids=selected["template_scope_id"].astype(str),
        extraction_uid=freeze["extraction_uid"],
        dataset_id="lfw",
        embedding_batch_size=int(
            CONFIG["gradcam"]["extraction"]["embedding_batch_size"]
        ),
        require_all_eligible=False,
    )
    coverage = prepared.loo_templates.coverage_summary()
    if len(prepared.sample_ids) != len(selected):
        raise RuntimeError("Pass A가 일부 선택 표본을 누락했습니다.")
    if WRITE_OUTPUTS:
        if PREPARED_ARTIFACT_OUTPUT_DIR is None:
            raise RuntimeError("PREPARED_ARTIFACT_OUTPUT_DIR를 지정하세요.")
        write_prepared_population_artifact(
            prepared,
            PREPARED_ARTIFACT_OUTPUT_DIR,
            shard_size=int(
                CONFIG["gradcam"]["extraction"]["shard_size"]
            ),
        )
else:
    coverage = pd.DataFrame(
        [{"status": "not_executed", "reason": "EXECUTE_STAGE=False"}]
    )
coverage


,template_scope_id,saliency_target_eligible,saliency_target_status,sample_count
0,calibration:49f688f0589b173e,False,singleton_identity,822
1,calibration:49f688f0589b173e,True,eligible,2217
2,development:49f688f0589b173e,False,singleton_identity,2408
3,development:49f688f0589b173e,True,eligible,5334
4,test:49f688f0589b173e,False,singleton_identity,832
5,test:49f688f0589b173e,True,eligible,1582


LFW의 singleton은 오류가 아니라 target 정의의 한계입니다. 임베딩은
유지하되 다른 target으로 바꾸지 않으며, 이후 집단 통계에서는 eligibility를
반드시 함께 보고합니다.
